# 04 Spark MLlib Models: Predicting Slow 311 Requests

**CSE 487 Phase 3, Team CTRL-Freaks** (MLlib models by Jayda Johnson)

**Purpose:** Re-implement two Phase 2 models with Spark MLlib, Logistic Regression and Random Forest, to predict whether a 311 request will be resolved slower than usual for its type, then compare the results with Phase 2.

**Inputs:** `workspace.ctrl_freaks.gold_311_features` (from `03_gold_layer`): closed requests, model features, the `isSlow` target, and a fixed 80/20 train/test `split` column.

**Outputs:** Saved Spark ML pipeline models in `/Volumes/workspace/ctrl_freaks/models/`, and a Phase 2 vs. Phase 3 comparison table.

**Assumptions:** `isSlow` uses training-only medians, all preprocessing is fit on training data only, and Random Forest is tuned with a validation split inside the training data, so the test set is used only once, for final scoring.

**Run instructions:** Run `01_bronze_layer`, `02_silver_layer`, and `03_gold_layer` first, then run this notebook from top to bottom.

Step 1: Stating the Prediction Task & Model Choice

**Task:** We are reusing the binary classification task from Project Phase 2 to predict `isSlow`, which outputs 1 if a request took longer than the median for its request category and 0 if not. We use two models from Phase 2 for this task: Logistic Regression and Random Forest.

**Why this task:** Binary Classification was chosen because it tests our questions about whether response times differ by request type or location, which helps explain why some requests get answered slower than others

**Why Logistic Regression:** Logistic Regression was picked because it outputs a probability between 0 and 1 that a 311 request will be slow, and its coefficients show which features of a request make it more likely to be slow.

**Why Random Forest:** Random Forest was the other model selected because it combines many decision trees, so it can catch combinations of features and its feature importances show whether request type or location matters more. 

**Step 2: Build the Spark ML Pipeline**

In [0]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, Imputer, VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier

In [0]:
#Logistic Regression part
lrCats = ["Neighborhood", "season", "Type", "Subject", "openDayOfWeek", "openHour","openMonth", "isWeekend", "Zipcode"]
lrNums = ["Latitude", "Longitude"]

lrIndexer = StringIndexer(inputCols=lrCats, outputCols=[c + "_idx" for c in lrCats], handleInvalid="keep")
lrEncoder = OneHotEncoder(inputCols=[c + "_idx" for c in lrCats], outputCols=[c + "_ohe" for c in lrCats])

lrImputer = Imputer(inputCols=lrNums, outputCols=[c + "_imp" for c in lrNums], strategy="median")
numAssembler = VectorAssembler(inputCols=[c + "_imp" for c in lrNums], outputCol="numVec")
lrScaler = StandardScaler(inputCol="numVec", outputCol="numScaled")

lrAssembler = VectorAssembler(inputCols=[c + "_ohe" for c in lrCats] + ["numScaled"], outputCol="features")
lr = LogisticRegression(featuresCol="features", labelCol="isSlow", maxIter=100)

lrPipeline = Pipeline(stages=[lrIndexer, lrEncoder, lrImputer, numAssembler, lrScaler, lrAssembler, lr])

In [0]:
#Random Forest part
rfCats = ["CouncilDistrict", "Type", "Subject", "season", "afterSwitch", "openYear", "openMonth"]

rfIndexer = StringIndexer(inputCols=rfCats, outputCols=[c + "_idx" for c in rfCats], handleInvalid="keep")
rfAssembler = VectorAssembler(inputCols=[c + "_idx" for c in rfCats], outputCol="features")

rf = RandomForestClassifier(featuresCol="features", labelCol="isSlow", numTrees=100, maxBins=64, seed=42)

rfPipeline = Pipeline(stages=[rfIndexer, rfAssembler, rf])

In [0]:
#quick check: lists each pipeline's steps in order
for name, pipe in [("Logistic Regression", lrPipeline), ("Random Forest", rfPipeline)]:
    print(name, "pipeline:")
    for i, stage in enumerate(pipe.getStages(), start=1):
        print("  ", i, type(stage).__name__)

Step 3: Train/Test Split



In [0]:
from pyspark.sql import functions as F

In [0]:
featuresTable = "workspace.ctrl_freaks.gold_311_features"
gold = spark.table(featuresTable)

train = gold.filter(F.col("split") == "train")
test = gold.filter(F.col("split") == "test")

total = gold.count()
trainRows = train.count()
testRows = test.count()
print("total:", total, "| train:", trainRows, f"({trainRows / total:.1%})", "| test:", testRows, f"({testRows / total:.1%})")

overlap = train.select("CaseReference").intersect(test.select("CaseReference")).count()
print("requests in both train and test:", overlap)

display(gold.groupBy("split", "isSlow").count().orderBy("split", "isSlow"))

Step 4: Training the Models

In [0]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

In [0]:
lrModel = lrPipeline.fit(train)
print("Logistic Regression trained")

In [0]:
fitPart, valPart = train.randomSplit([0.8, 0.2], seed=42)
aucEval = BinaryClassificationEvaluator(labelCol="isSlow", rawPredictionCol="rawPrediction", metricName="areaUnderROC")

rf.setNumTrees(50)
bestDepth, bestAuc = None, -1
for depth in [5, 10]:
    rf.setMaxDepth(depth)
    candidate = rfPipeline.fit(fitPart)
    valAuc = aucEval.evaluate(candidate.transform(valPart))
    print("maxDepth", depth, "| validation AUC", round(valAuc, 4))
    if valAuc > bestAuc:
        bestDepth, bestAuc = depth, valAuc

rf.setMaxDepth(bestDepth)
rfModel = rfPipeline.fit(train)
print("Random Forest trained, best maxDepth:", bestDepth)

Step 5: Evaluation Metrics 

In [0]:
def evaluateModel(predictions, modelName):
    multi = MulticlassClassificationEvaluator(labelCol="isSlow", predictionCol="prediction")
    auc = BinaryClassificationEvaluator(labelCol="isSlow", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
    results = {
        "model": modelName,
        "accuracy": round(multi.evaluate(predictions, {multi.metricName: "accuracy"}), 4),
        # precision, recall, and F1 for the slow class (isSlow = 1)
        "precision": round(multi.evaluate(predictions, {multi.metricName: "precisionByLabel", multi.metricLabel: 1.0}), 4),
        "recall": round(multi.evaluate(predictions, {multi.metricName: "recallByLabel", multi.metricLabel: 1.0}), 4),
        "f1": round(multi.evaluate(predictions, {multi.metricName: "fMeasureByLabel", multi.metricLabel: 1.0}), 4),
        "auc": round(auc.evaluate(predictions), 4),
    }
    print(results)
    display(predictions.groupBy("isSlow").pivot("prediction").count().orderBy("isSlow"))
    return results

lrPreds = lrModel.transform(test)
lrResults = evaluateModel(lrPreds, "Logistic Regression")

rfPreds = rfModel.transform(test)
rfResults = evaluateModel(rfPreds, "Random Forest")

Step 6: Comparison with Phase 2

In [0]:
import pandas as pd

#Phase 2 results (scikit-learn), from our Phase 2 report; Phase 2 didn't report AUC for Random Forest
phase2 = [
    {"model": "Logistic Regression", "phase": "Phase 2 (sklearn)", "accuracy": 0.575, "precision": 0.58, "recall": 0.53, "f1": 0.56, "auc": 0.605},
    {"model": "Random Forest", "phase": "Phase 2 (sklearn)", "accuracy": 0.660, "precision": 0.67, "recall": 0.63, "f1": 0.65, "auc": None},
]

#Phase 3 results (Spark MLlib) from the evaluation cell
phase3 = [dict(r, phase="Phase 3 (Spark)") for r in [lrResults, rfResults]]

comparison = pd.DataFrame(phase2 + phase3)[["model", "phase", "accuracy", "precision", "recall", "f1", "auc"]]
display(comparison.sort_values(["model", "phase"]).reset_index(drop=True))

**How Phase 3 compares to Phase 2:**

- **Logistic Regression** stayed almost the same (57.3% vs. 57.5% accuracy, AUC 0.604 vs. 0.605). Its setup barely changed (same features, same train-only slow label, same 80/20 ratio), so this shows Spark reproduced our Phase 2 model. The small gap comes from a different set of train/test rows and slight differences between Spark's and scikit-learn's preprocessing.

- **Random Forest** dropped from 66.0% to 64.2% accuracy, but it is still our best model (AUC 0.702). We expected a small drop because Phase 3 fixed two problems from Phase 2: the slow label now uses medians from training rows only (Phase 2's used all rows, which leaked test information), and tree depth is now picked with a validation split instead of the test set. Phase 3 also uses 50 trees and a maximum depth of 10 (Phase 2: 100 trees, depth 20) to fit Free Edition's compute limits. So Phase 2's 66% was slightly optimistic, and 64.2% is a more honest estimate.

Step 7: Save and Load the Models

In [0]:
import os
from pyspark.ml import PipelineModel

# make a volume (storage folder) for the models if it doesn't exist yet
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.ctrl_freaks.models")
modelDir = "/Volumes/workspace/ctrl_freaks/models"
os.environ["SPARKML_TEMP_DFS_PATH"] = modelDir   # Serverless needs this set to a volume for Spark ML

# save each fitted pipeline (prep steps + model together)
lrPath = modelDir + "/logistic_regression"
rfPath = modelDir + "/random_forest"
lrModel.write().overwrite().save(lrPath)
rfModel.write().overwrite().save(rfPath)
print("saved:", lrPath, "and", rfPath)

# load them back and check they give the same predictions
lrLoaded = PipelineModel.load(lrPath)
rfLoaded = PipelineModel.load(rfPath)
lrCheck = evaluateModel(lrLoaded.transform(test), "Logistic Regression (loaded)")
rfCheck = evaluateModel(rfLoaded.transform(test), "Random Forest (loaded)")
print("loaded models match:", lrCheck["accuracy"] == lrResults["accuracy"] and rfCheck["accuracy"] == rfResults["accuracy"])